# 1. Objective

The goal of this notebook is to design the extraction pipeline that converts one raw Slay the Spire run into the canonical decision dataset defined in Notebook 01.

# 2. Run Inspection

Loading an Ironclad run from the raw dataset.

In [162]:
# Reload raw run data (each notebook has its own kernel/session)
import gzip
import json

with gzip.open("../data/raw/2020-09-28-04-45_965.json.gz", "rt", encoding="utf-8") as f:
    runs = json.load(f)

# Filter to Ironclad runs, per DD-003
ironclad_runs = [r['event'] for r in runs if r['event'].get('character_chosen') == 'IRONCLAD']

# Filter to won Ironclad runs, 
won_ironclad_runs = [r for r in ironclad_runs if r.get('victory') == True]

# Inspect how many Ironclad runs exist in the raw data
print(f"Number of Ironclad runs: {len(ironclad_runs)}")
print(f"Number of won Ironclad runs: {len(won_ironclad_runs)}")

Number of Ironclad runs: 342
Number of won Ironclad runs: 31


Inspection of the main characteristics of a run as specified in the dataset schema in Notebook 1.

In [163]:
sample_run = ironclad_runs[0]

print(sample_run.keys())
print("Card Choices:", sample_run['card_choices'])
print("Master Deck:", sample_run['master_deck'])
print("Relics:", sample_run['relics'])
print("Current HP per Floor:", sample_run['current_hp_per_floor'])
print(type(sample_run['current_hp_per_floor']))
print("Max HP per Floor:", sample_run['max_hp_per_floor'])
print("Gold per Floor:", sample_run['gold_per_floor'])
print("Floor Reached:", sample_run['floor_reached'])


dict_keys(['gold_per_floor', 'floor_reached', 'playtime', 'items_purged', 'score', 'play_id', 'local_time', 'is_ascension_mode', 'campfire_choices', 'neow_cost', 'seed_source_timestamp', 'circlet_count', 'master_deck', 'relics', 'potions_floor_usage', 'damage_taken', 'seed_played', 'potions_obtained', 'is_trial', 'path_per_floor', 'character_chosen', 'items_purchased', 'campfire_rested', 'item_purchase_floors', 'current_hp_per_floor', 'gold', 'neow_bonus', 'is_prod', 'is_daily', 'chose_seed', 'campfire_upgraded', 'win_rate', 'timestamp', 'path_taken', 'build_version', 'purchased_purges', 'victory', 'max_hp_per_floor', 'card_choices', 'player_experience', 'relics_obtained', 'event_choices', 'is_beta', 'boss_relics', 'items_purged_floors', 'is_endless', 'potions_floor_spawned', 'killed_by', 'ascension_level'])
Card Choices: [{'not_picked': ['Clothesline', 'Rage', 'Infernal Blade'], 'picked': 'SKIP', 'floor': 1}, {'not_picked': ['Thunderclap', 'Pummel', 'Searing Blow'], 'picked': 'SKIP', 

The most important fields are the following:
- 'card_choises': a disctionary which contains all the offered cards and the chosed card or 'SKIP'
- 'master_deck': a list which contains the whole deck at the end of the run 
- 'relics': a list of all the relics accumulated by the end of the run
- 'current_hp_per_floor': a list containing the hp of the player on every floor
- 'max_hp_per_floor': a list containing the max hp of the player on every floor
- 'gold_per_floor': a list containing the current gold of the player on every floor
- 'floor_reached': The floor that the player finished his run

The inspection provided an overview of the available information. Before designing the extraction algorithm, the reliability and consistency of these fields must be verified.

# 3. Dataset Validation

## Finding F-001 — 'current_hp_per_floor' is not a reliable floor index

### Question: Does current_hp_per_floor correspond one-to-one with floors?

Noticed an inconsistency between the number of current_hp_per_floor and floor_reached, which is worth investigating. A likely explanation is that it is a special seed or some other peculiarity.

In [164]:
#Inspect the keys of a lost Ironclad run

print("Is Trial:", sample_run['is_trial'])
print("Is Daily:", sample_run['is_daily'])
print("Chose Seed:", sample_run['chose_seed'])
print("Path Taken:", sample_run['path_taken'])
print("Killed By:", sample_run['killed_by'])

print("Number of times Hp per floor is stored in a lost run:", len(sample_run['current_hp_per_floor']))
print("Floor Reached:", sample_run['floor_reached'])


Is Trial: False
Is Daily: False
Chose Seed: False
Path Taken: ['M', 'M', '?', '$', '?']
Killed By: 2 Louse
Number of times Hp per floor is stored in a lost run: 6
Floor Reached: 5


In [165]:
#Inspect the keys of a won Ironclad run

sample_run_2 = won_ironclad_runs[0]

print(sample_run_2.keys())
print("Card Choices:",sample_run_2['card_choices'])
print("Master Deck:",sample_run_2['master_deck'])
print("Relics:",sample_run_2['relics'])
print("Current HP per Floor:",sample_run_2['current_hp_per_floor'])
print("Max HP per Floor:",sample_run_2['max_hp_per_floor'])
print("Gold per Floor:",sample_run_2['gold_per_floor'])
print("Floor Reached:",sample_run_2['floor_reached'])   
print("Is Trial:",sample_run_2['is_trial'])
print("Is Daily:",sample_run_2['is_daily'])
print("Chose Seed:",sample_run_2['chose_seed'])
print("Path Taken:",sample_run_2['path_taken'])


print("Number of times Hp per floor is stored in a won run:", len(sample_run_2['current_hp_per_floor']))
print("Floor Reached:", sample_run_2['floor_reached'])


dict_keys(['gold_per_floor', 'floor_reached', 'playtime', 'items_purged', 'score', 'play_id', 'local_time', 'is_ascension_mode', 'campfire_choices', 'neow_cost', 'seed_source_timestamp', 'circlet_count', 'master_deck', 'special_seed', 'relics', 'potions_floor_usage', 'damage_taken', 'seed_played', 'potions_obtained', 'is_trial', 'path_per_floor', 'character_chosen', 'items_purchased', 'campfire_rested', 'item_purchase_floors', 'current_hp_per_floor', 'gold', 'neow_bonus', 'is_prod', 'is_daily', 'chose_seed', 'campfire_upgraded', 'win_rate', 'timestamp', 'path_taken', 'build_version', 'purchased_purges', 'victory', 'max_hp_per_floor', 'card_choices', 'player_experience', 'relics_obtained', 'event_choices', 'is_beta', 'boss_relics', 'items_purged_floors', 'is_endless', 'potions_floor_spawned', 'ascension_level'])
Card Choices: [{'not_picked': ['Heavy Blade', 'Sword Boomerang'], 'picked': 'Flex', 'floor': 1.0}, {'not_picked': ['Pommel Strike', 'Shrug It Off'], 'picked': 'Whirlwind', 'floo

**Result:** there is an inconcistency between the lost run and the won run. In the lost run :<br>
len(current_hp_per_floor) = floor_reached + 1 <br>
In the won run:<br>
len(current_hp_per_floor) = floor_reached - 1 <br>
That's an actual 2 entry swing between cases which means there isn't a single simple offset rule (like "always add 1 for starting HP").

In [166]:
#Compare the lengths of current_hp_per_floor and path_per_floor for lost and won runs

print("Length of current_hp_per_floor for lost runs:", len(sample_run['current_hp_per_floor']))
print("Floor Reached for lost runs:", sample_run['floor_reached'])
print("Length of path_per_floor for lost runs:", len(sample_run['path_per_floor']))

print("Length of current_hp_per_floor for won runs:", len(sample_run_2['current_hp_per_floor']))
print("Floor Reached for won runs:", sample_run_2['floor_reached'])
print("Length of path_per_floor for won runs:", len(sample_run_2['path_per_floor']))

Length of current_hp_per_floor for lost runs: 6
Floor Reached for lost runs: 5
Length of path_per_floor for lost runs: 5
Length of current_hp_per_floor for won runs: 50
Floor Reached for won runs: 51
Length of path_per_floor for won runs: 51


We see that 'path_per_floor' matches 'floor_reached' in both cases (5 and 5 for the death run, 51 and 51 for the victory run). So 'path_per_floor' is the stable, trustworthy reference: one entry per floor actually visited, consistent regardless of how the run ended.

### Decision : `current_hp_per_floor` indexing is inconsistent

Comparing against `path_per_floor` (which reliably matches `floor_reached` in both examples tested) revealed that `current_hp_per_floor`'s length doesn't align consistently:
- Death run (floor_reached=5): 6 entries (one extra).
- Victory run (floor_reached=51): 50 entries (one short).

`path_per_floor` will be used as the reliable floor-count reference. HP alignment for the final floor of a run is treated as unreliable and will be handled explicitly in Section 3.

## Finding F-002: 'killed_by' key value is missing from the won run.

### Question: Which keys are inconsistently present across runs?

In [167]:
# Compare the number of keys in lost and won runs, and check for the presence of 'killed_by' key

print("Number of keys in lost runs:", len(sample_run.keys()))
print("killed_by in lost runs:", 'killed_by' in sample_run)

print("Number of keys in won runs:", len(sample_run_2.keys()))
print("killed_by in won runs:", 'killed_by' in sample_run_2)

Number of keys in lost runs: 49
killed_by in lost runs: True
Number of keys in won runs: 49
killed_by in won runs: False


In [168]:
#Find the keys that are present in one run but not the other

death_keys = set(sample_run.keys())
victory_keys = set(sample_run_2.keys())

print("In death run but not victory run:", death_keys - victory_keys)
print("In victory run but not death run:", victory_keys - death_keys)

In death run but not victory run: {'killed_by'}
In victory run but not death run: {'special_seed'}


In [169]:
# Does the presence of a special seed correlate with victory? Let's check.

has_special_seed = [('special_seed' in r['event']) for r in runs]
print(sum(has_special_seed), "out of", len(runs))

# cross-check against victory, to see if there's a real relationship or not
import collections
victory_by_has_field = collections.Counter(
    (r['event'].get('victory'), 'special_seed' in r['event']) for r in runs
)
print(victory_by_has_field)

323 out of 965
Counter({(False, False): 600, (False, True): 279, (True, True): 44, (True, False): 42})


**Result:** `special_seed` is missing entirely (not just False) in ~33% of runs (323/965). No relationship found with `victory`, likely an unrelated data logging inconsistency.

In [170]:
#Counting the number of times each key appears in the runs, to see if there are any keys that are not present in all runs

import collections

key_counts = collections.Counter()
for r in runs:
    key_counts.update(r['event'].keys())

total_runs = len(runs)
for key, count in sorted(key_counts.items(), key=lambda x: x[1]):
    if count < total_runs:
        print(f"{key}: present in {count}/{total_runs} ({count/total_runs:.1%})")

special_seed: present in 323/965 (33.5%)
killed_by: present in 753/965 (78.0%)


### Decision: Filtering logic must use `.get('special_seed', False)` rather than direct indexing to avoid KeyErrors.

## Finding F-003: There exist runs that were abandoned

### Question: Are there runs that resulted in a loss that didn't finish due to a 'killed_by'?


In [171]:
#Checking how many runs are victories

victories = sum(1 for r in runs if r['event'].get('victory'))
print(victories, "victories out of", len(runs))

86 victories out of 965


Our hypothesis of 'killed_by' missing only from the won runs is not correct since the numbers 86 and 965-753=212 do not allign. This is an actual discrepancy and not just noise. We use the same technique as with the 'special_seed' cross-tab, to see from which runs the key value 'killed_by' is missing.

In [172]:
#Counting how many runs are killed by something, and cross-checking against victory

import collections
killed_by_crosstab = collections.Counter(
    (r['event'].get('victory'), 'killed_by' in r['event']) for r in runs
)
print(killed_by_crosstab)

Counter({(False, True): 753, (False, False): 126, (True, False): 86})


Notice there's no (True, True) combination at all, so no victory run has 'killed_by'. Since 86 'killed_by' are missing from the Victories the question is where the rest 126 are missnig from. A likely candidate would be: runs abandoned/quit before dying or winning.

In [173]:
#Counting how many runs are abandoned (i.e., not victorious and not killed by anything)

abandoned_candidates = [r['event'] for r in runs 
                         if r['event'].get('victory') == False 
                         and 'killed_by' not in r['event']]

print(len(abandoned_candidates))

#Check the first abandoned candidate for floor_reached, path_per_floor, and card_choices
print(abandoned_candidates[0].get('floor_reached'))
print(abandoned_candidates[0].get('path_per_floor'))
print(abandoned_candidates[0].get('card_choices'))

126
0
[]
[]


In [174]:
#Check the floor_reached values for all abandoned candidates

floor_reached_values = [r.get('floor_reached') for r in abandoned_candidates]
print(set(floor_reached_values))

{0, 2, 3, 4, 5, 6, 7, 8, 11, 13, 14, 15, 17, 19, 20, 21, 22, 30}


**Result**: `floor_reached` values for the 126 non-death, non-victory runs range from 0 to 30, confirming these are not all trivial floor-0 quits. Many contain real `card_choices` made under normal play conditions before the player stopped for unrelated reasons.

### Decision: flag missing keys via `is_abandoned` (`victory == False AND 'killed_by' not in run`). Retain them in the canonical and behavioral (Phase 3) dataset, since the decisions themselves are valid signal. Exclude them from any future model using `victory` as a label or sample weight, since an abandoned run's outcome reflects the player stopping, not decision quality. Tracked as DD-011 in Notebook 01.

## Finding F-004 Can the deck be reconstructed exactly at each card choice?

##
| Finding    | Status | Action    |
| :---        |    :----:   |          ---: | 
| F-001 | Confirmed | Use 'path_per_floor' | 
| F-002 | Confirmed | Use .get() for optional keys | 
| F-003 | Confirmed | Keep abandoned runs with flag | 
| F-004 | Open | Pending | 


# 4. Timeline reconstruction

# 5. Pseudocode

# 6. Exploratory implementation

# 7. Manual validation

# 8. Open questions